# Part D Schritt fuer Schritt

Dieses Notebook zeigt den bereits vorhandenen Part-D-Code anschaulich.

Part D baut auf Part B und C auf:

```text
raw image -> Part B preprocess -> Part C road mask -> allowed region -> patches -> Conv-AE
```

Was hier sichtbar gemacht wird:
- Strassenmaske laden oder erzeugen
- sichere Patch-Zentren aus `allowed_region()` bestimmen
- Tiefenbaender ueber die Strasse legen
- konkrete 128x128-Patches anzeigen
- vorhandene Stage-1-Shards und Index-Dateien inspizieren
- vorhandenen Stage-1-Trainingsoutput anzeigen, falls schon gerechnet

Teure Schritte wie `extract` und `train` sind optional und werden nicht automatisch gestartet.

In [ ]:
from pathlib import Path
import sys
import time

ROOT = Path.cwd()
if not (ROOT / "aibip").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import cv2
import numpy as np

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except NameError:
    pass

import matplotlib.pyplot as plt
from IPython.display import display, Image

from aibip import config, imgio
from aibip.part_b import preprocess
from aibip.part_c import allowed_region, segment_road
from aibip.part_d import (
    patch_center_seed,
    band_edges,
    alloc_budget,
    plan_centers,
    shard_paths,
    read_index,
    split_ids,
    select_rows,
)

plt.rcParams["figure.figsize"] = (14, 7)
plt.rcParams["axes.titlesize"] = 10

print("Repo:", ROOT)
print("PATCH:", config.PATCH)
print("N_BANDS:", config.N_BANDS)
print("D_CAP_PER_IMAGE:", config.D_CAP_PER_IMAGE)
print("Stage-1 data:", ROOT / config.D_DATA_DIR[1])
print("Stage-2 data:", ROOT / config.D_DATA_DIR[2])

## 1. Bilder laden

Wie in den anderen Notebooks werden zuerst die Beispielbilder aus `data/mini` genommen.

In [ ]:
EXAMPLE_PATHS = [
    Path("data/mini/WE1MY5SK.jpg"),
    Path("data/mini/WE1MY5SL.jpg"),
    Path("data/mini/WE1MY5SP.jpg"),
    Path("data/mini/WE1MY5SN.jpg"),
    Path("data/mini/WE1MY5SM.jpg"),
    Path("data/mini/WE1MY5SR.jpg"),
    Path("data/mini/WE1MY5SQ.jpg"),
    Path("data/mini/WE1MY5SO.jpg"),
    Path("data/mini/WE1MY5SS.jpg"),
    Path("data/mini/WE1MY5ST.jpg"),
]

paths = [p for p in EXAMPLE_PATHS if p.exists()]
if not paths:
    fallback = ROOT / "data" / "mini"
    paths = imgio.list_images(fallback)[:10] if fallback.exists() else []
if not paths:
    raise FileNotFoundError("Keine Beispielbilder gefunden. Lege Bilder in data/mini oder passe EXAMPLE_PATHS an.")

raw = {p.stem: imgio.load_image(p) for p in paths}
print("Gefundene Bilder:", len(raw))
for p in paths:
    print("-", p)

In [ ]:
def bgr_to_rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def show_images(items, cols=3, title=None):
    rows = int(np.ceil(len(items) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 3.4 * rows), squeeze=False)
    for ax in axes.ravel():
        ax.axis("off")
    for ax, (name, img) in zip(axes.ravel(), items):
        ax.imshow(bgr_to_rgb(img))
        ax.set_title(name)
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    display(fig)
    plt.close(fig)

show_images([(name, imgio.resize(img, (384, 512))) for name, img in list(raw.items())[:6]], cols=3, title="Rohbilder")

## 2. Part-B-Input fuer Part D

Part D arbeitet nicht mit dem Rohbild, sondern mit `preprocess(raw)`: zugeschnitten, verkleinert und mit CLAHE normalisiert.

In [ ]:
processed = {name: preprocess(img) for name, img in raw.items()}
for name, proc in list(processed.items())[:3]:
    print(name, proc.shape)

show_images([(name, imgio.resize(proc, (307, 1024))) for name, proc in list(processed.items())[:6]], cols=2, title="preprocess(raw): das Arbeitsbild fuer C/D")

## 3. Strassenmaske bereitstellen

Part D braucht die Maske aus Part C. Diese Zelle nutzt zuerst `outputs/masks/<image_id>.png`. Wenn keine Maske existiert, kannst du `COMPUTE_MISSING_MASK = True` setzen. Dann wird fuer ein Bild MobileSAM ausgefuehrt.

Falls kein Modell vorhanden ist oder du nicht warten willst, erzeugt die Zelle eine klar markierte Demo-Maske. Diese ist nur zur Erklaerung der Patch-Logik gedacht, nicht als echtes Ergebnis.

In [ ]:
COMPUTE_MISSING_MASK = False  # bei Bedarf auf True setzen; auf CPU kann SAM langsam sein

def demo_road_mask(shape):
    h, w = shape[:2]
    poly = np.array([
        [int(0.46 * w), int(0.15 * h)],
        [int(0.56 * w), int(0.15 * h)],
        [int(0.84 * w), int(0.98 * h)],
        [int(0.16 * w), int(0.98 * h)],
    ], dtype=np.int32)
    mask = np.zeros((h, w), dtype=np.uint8)
    cv2.fillPoly(mask, [poly], 1)
    return mask.astype(bool)

def load_or_make_mask(name, raw_img, proc):
    mask_path = ROOT / config.MASKS_DIR / f"{name}.png"
    if mask_path.exists():
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE) > 0
        if mask.shape != proc.shape[:2]:
            mask = cv2.resize(mask.astype(np.uint8), (proc.shape[1], proc.shape[0]), interpolation=cv2.INTER_NEAREST).astype(bool)
        return mask, "geladen aus outputs/masks"
    if COMPUTE_MISSING_MASK:
        return segment_road(raw_img), "neu berechnet mit MobileSAM"
    return demo_road_mask(proc.shape), "DEMO-MASKE, nicht Part-C-Ergebnis"

name = list(raw.keys())[0]
proc = processed[name]
mask, mask_source = load_or_make_mask(name, raw[name], proc)
print(name, mask.shape, mask_source, "mask_frac=", round(float(mask.mean()), 4))

## 4. Maske und Allowed Region anzeigen

Die Strassenmaske zeigt, was als Strasse gilt. Die Allowed Region ist kleiner: Dort duerfen spaeter Patch-Zentren liegen. Der Rand wird entfernt, damit ein 128x128-Patch nicht halb auf Gehweg, Bordstein oder Vegetation liegt.

In [ ]:
def mask_overlay(proc, mask, color=(0, 255, 0), alpha=0.45):
    out = proc.copy()
    c = np.array(color, dtype=np.float32)
    out[mask] = (out[mask] * (1 - alpha) + c * alpha).astype(np.uint8)
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(out, contours, -1, color, 2)
    return out

allowed = allowed_region(mask)
combined = proc.copy()
combined[mask] = (combined[mask] * 0.65 + np.array([0, 255, 0]) * 0.35).astype(np.uint8)
combined[allowed] = (combined[allowed] * 0.40 + np.array([255, 0, 0]) * 0.60).astype(np.uint8)

print("Maskenflaeche:", round(float(mask.mean()), 4))
print("Allowed-Region-Flaeche:", round(float(allowed.mean()), 4))
print("Interpretation: blau = sichere Patch-Zentren, gruen = Strasse allgemein")
show_images([
    ("Strassenmaske", imgio.resize(mask_overlay(proc, mask), (307, 1024))),
    ("Allowed Region", imgio.resize(combined, (307, 1024))),
], cols=1)

## 5. Tiefenbaender

Part D nutzt keine Bird's-Eye-View-Transformation. Stattdessen wird die Strasse in `N_BANDS = 4` Tiefenbaender geteilt: oben = fern, unten = nah. Dadurch kann spaeter pro Band anders bewertet werden, weil ferne Strasse im Bild kleiner und unschaerfer erscheint.

In [ ]:
road_rows = np.nonzero(mask.any(axis=1))[0]
edges = band_edges(road_rows)
print("Band edges:", edges.tolist(), "(0=far,", config.N_BANDS - 1, "=near)")

band_view = proc.copy()
colors = [(255, 80, 80), (80, 180, 255), (80, 255, 120), (255, 220, 80)]
for band in range(config.N_BANDS):
    y0, y1 = int(edges[band]), int(edges[band + 1])
    band_mask = mask.copy()
    band_mask[:y0] = False
    band_mask[y1:] = False
    color = np.array(colors[band], dtype=np.float32)
    band_view[band_mask] = (band_view[band_mask] * 0.45 + color * 0.55).astype(np.uint8)
    cv2.putText(band_view, f"band {band}", (20, max(30, y0 + 30)), cv2.FONT_HERSHEY_SIMPLEX, 1.0, colors[band], 3)

show_images([("Tiefenbaender ueber der Strassenmaske", imgio.resize(band_view, (307, 1024)))], cols=1)

## 6. Patch-Zentren planen

`plan_centers(mask, image_id)` bestimmt bis zu `D_CAP_PER_IMAGE = 24` Patch-Zentren. Die Auswahl ist deterministisch: gleicher `image_id` und gleicher Seed ergeben dieselben Zentren.

In [ ]:
centers = plan_centers(mask, name)
print("Seed fuer dieses Bild:", patch_center_seed(name))
print("Geplante Patch-Zentren:", len(centers))
print("Pro Band:", np.bincount([c["band"] for c in centers], minlength=config.N_BANDS).tolist() if centers else [])
centers[:5]

In [ ]:
def draw_patch_centers(proc, centers):
    out = proc.copy()
    r = config.PATCH // 2
    colors = [(255, 80, 80), (80, 180, 255), (80, 255, 120), (255, 220, 80)]
    for c in centers:
        row, col, band = c["row"], c["col"], c["band"]
        color = colors[band]
        cv2.rectangle(out, (col - r, row - r), (col + r, row + r), color, 2)
        cv2.circle(out, (col, row), 4, color, -1)
    return out

center_view = draw_patch_centers(mask_overlay(proc, mask, alpha=0.22), centers)
show_images([("128x128-Patchfenster mit Zentren", imgio.resize(center_view, (307, 1024)))], cols=1)

## 7. Die tatsaechlichen Patches ansehen

Hier werden aus dem vorbereiteten Bild die 128x128-Ausschnitte ausgeschnitten, die spaeter in den Autoencoder gehen.

In [ ]:
def crop_patch(proc, center):
    r = config.PATCH // 2
    y, x = center["row"], center["col"]
    return proc[y - r:y + r, x - r:x + r]

patches = [(f"band {c['band']} row {c['row']}", crop_patch(proc, c)) for c in centers[:12]]
if patches:
    show_images(patches, cols=4, title="Beispiel-Patches fuer den Autoencoder")
else:
    print("Keine Patch-Zentren gefunden. Bei echten Masken kann das auf eine zu kleine Allowed Region hindeuten.")

## 8. Budget-Verteilung demonstrieren

Die Funktion `alloc_budget()` verteilt das Patch-Budget proportional zur erlaubten Flaeche pro Band. Breite/nahe Bereiche bekommen meist mehr Patches, aber die Gesamtzahl bleibt gedeckelt.

In [ ]:
rows, cols = np.nonzero(allowed)
if rows.size:
    band_id = np.searchsorted(edges, rows, side="right") - 1
    weights = np.bincount(band_id, minlength=config.N_BANDS)
    budget = alloc_budget(weights, config.D_CAP_PER_IMAGE)
    for band, (weight, n) in enumerate(zip(weights, budget)):
        print(f"band {band}: allowed pixels={int(weight):6d}, budget={int(n):2d}")
else:
    print("Allowed Region ist leer.")

## 9. Vorhandene Stage-1-Shards inspizieren

Wenn du bereits `part_d extract --stage 1` ausgefuehrt hast, liegen hier ein Memmap-Shard und eine Index-CSV. Diese Zelle laedt nur Metadaten und eine kleine Stichprobe.

In [ ]:
stage = 1
shard_path, index_path = [ROOT / p for p in shard_paths(stage)]
print("Shard:", shard_path)
print("Index:", index_path)

if shard_path.exists() and index_path.exists():
    rows = read_index(index_path)
    shard = np.load(shard_path, mmap_mode="r")
    print("Shard shape:", shard.shape, shard.dtype)
    print("Index rows:", len(rows))
    print("Unique images:", len({r['image_id'] for r in rows}))
    print("Per band:", np.bincount([r['band'] for r in rows], minlength=config.N_BANDS).tolist())
else:
    rows = []
    shard = None
    print("Noch keine Stage-1-Shards vorhanden.")
    print("Zum Erzeugen im Terminal:")
    print(r".\.venv\Scripts\python.exe -m aibip.part_c run data\mini")
    print(r".\.venv\Scripts\python.exe -m aibip.part_d extract --stage 1")

In [ ]:
if shard is not None and rows:
    rng = np.random.default_rng(config.SEED)
    picks = sorted(rng.choice(len(rows), size=min(12, len(rows)), replace=False).tolist())
    gallery = []
    for idx in picks:
        rec = rows[idx]
        patch = np.ascontiguousarray(shard[rec["patch_idx"]])
        gallery.append((f"{rec['image_id']} b{rec['band']}", patch))
    show_images(gallery, cols=4, title="Patches aus dem gespeicherten Stage-1-Shard")
else:
    print("Keine gespeicherten Patches zum Anzeigen.")

## 10. Train/Val-Split nach Bild-ID

Wichtig fuer Part D: Der Split passiert nach `image_id`, nicht zufaellig nach Patch. Sonst wuerden fast gleiche Patches desselben Bildes gleichzeitig in Training und Validierung landen.

In [ ]:
if rows:
    ids = sorted({r["image_id"] for r in rows})
    train_ids, val_ids = split_ids(ids)
    train_rows = select_rows(rows, train_ids)
    val_rows = select_rows(rows, val_ids)
    print("Images train/val:", len(train_ids), len(val_ids))
    print("Patches train/val:", len(train_rows), len(val_rows))
    print("Overlap:", set(train_ids) & set(val_ids))
else:
    demo_ids = [f"IMG_{i:02d}" for i in range(20)]
    train_ids, val_ids = split_ids(demo_ids)
    print("Demo-Split ohne Shard:")
    print("train:", train_ids[:5], "...", len(train_ids))
    print("val:", val_ids, len(val_ids))

## 11. Vorhandenen Trainingsoutput ansehen

Wenn `part_d train --stage 1` schon gelaufen ist, zeigt diese Zelle die Loss-Kurve und das Rekonstruktionsgrid aus `docs/assets`.

In [ ]:
loss_png = ROOT / config.FIGURES_DIR / "d_loss_curve_stage1.png"
recon_png = ROOT / config.FIGURES_DIR / "d_recon_grid_stage1.png"

for path in [loss_png, recon_png]:
    print(path, "exists=", path.exists())
    if path.exists():
        display(Image(filename=str(path)))

if not loss_png.exists() or not recon_png.exists():
    print("Zum Erzeugen:")
    print(r".\.venv\Scripts\python.exe -m aibip.part_d train --stage 1")

## 12. Optionale Kommandos fuer Part D

Diese Kommandos laufen im Terminal, nicht automatisch im Notebook:

```powershell
cd road-anomaly-detection

# Voraussetzung: Part-C-Masken fuer data/mini existieren
.\.venv\Scripts\python.exe -m aibip.part_c run data\mini

# Stage 1: Mini-Dry-Run
.\.venv\Scripts\python.exe -m aibip.part_d extract --stage 1
.\.venv\Scripts\python.exe -m aibip.part_d train --stage 1
.\.venv\Scripts\python.exe -m aibip.part_d preview --stage 1 --n 6

# Stage 2: voller Pool, gleicher Codepfad, nur groessere Daten
.\.venv\Scripts\python.exe -m aibip.part_c run "data\26-07-14 Cyclomedia-Test-Hammersbach\panoramas"
.\.venv\Scripts\python.exe -m aibip.part_d extract --stage 2
.\.venv\Scripts\python.exe -m aibip.part_d train --stage 2
```

Stage 1 und Stage 2 nutzen dieselbe Logik. Der Unterschied steckt in `config.D_DATA_DIR`, `config.D_EPOCHS` und `config.D_BATCH`.